In [ ]:
!curl http://localhost:8003/v1/models

{"object":"list","data":[{"id":"google/medgemma-27b-it","object":"model","created":1771591349,"owned_by":"vllm","root":"google/medgemma-27b-it","parent":null,"max_model_len":8192,"permission":[{"id":"modelperm-383cd7e6f48c4487830ab547fa456f8b","object":"model_permission","created":1771591349,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [ ]:
import os
from openai import OpenAI

input_folder = "../input_temp"
output_folder = "output_temp"

# Sicherstellen, dass Output-Ordner existiert
os.makedirs(output_folder, exist_ok=True)

# Client
client = OpenAI(
    base_url="http://localhost:8003/v1",
    api_key="dummy"
)

system_prompt = "Du bist eine KI, die Nutzern hilft transkribierte Texte weiterzuverarbeiten und aufzubereiten. Du bekommst ein Transkript. Dieses sollst du zusammenfassen."

# Alle Dateien im Input-Ordner
for filename in os.listdir(input_folder):
    if filename.endswith(".txt"):
        input_path = os.path.join(input_folder, filename)

        # Datei lesen
        with open(input_path, "r", encoding="utf-8") as f:
            user_prompt = f.read()

        print(f"Verarbeite: {filename}")

        # 3 Durchläufe
        for i in range(1, 4):
            try:
                response = client.chat.completions.create(
                    model="google/medgemma-27b-it",
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    temperature=0.0,
                    top_p=0.5
                    
                )

                output_text = response.choices[0].message.content

                # Output-Datei benennen
                base_name = os.path.splitext(filename)[0]
                output_filename = f"{base_name}_run{i}.txt"
                output_path = os.path.join(output_folder, output_filename)

                # Speichern
                with open(output_path, "w", encoding="utf-8") as f:
                    f.write(output_text)

                print(f"  → Gespeichert: {output_filename}")

            except Exception as e:
                print(f"  ❌ Fehler bei Run {i} von {filename}: {e}")

Verarbeite: Anam_Aengstliche-Patientin.txt
  → Gespeichert: Anam_Aengstliche-Patientin_run1.txt
  → Gespeichert: Anam_Aengstliche-Patientin_run2.txt
  → Gespeichert: Anam_Aengstliche-Patientin_run3.txt
Verarbeite: Anam_Pneumonie_02.txt
  → Gespeichert: Anam_Pneumonie_02_run1.txt
  → Gespeichert: Anam_Pneumonie_02_run2.txt
  → Gespeichert: Anam_Pneumonie_02_run3.txt
Verarbeite: Anam_sexueller Missbrauch.txt
  → Gespeichert: Anam_sexueller Missbrauch_run1.txt
  → Gespeichert: Anam_sexueller Missbrauch_run2.txt
  → Gespeichert: Anam_sexueller Missbrauch_run3.txt
Verarbeite: Anam_Gastroenteritis.txt
  → Gespeichert: Anam_Gastroenteritis_run1.txt
  → Gespeichert: Anam_Gastroenteritis_run2.txt
  → Gespeichert: Anam_Gastroenteritis_run3.txt
Verarbeite: Anam_Lymphom.txt
  → Gespeichert: Anam_Lymphom_run1.txt
  → Gespeichert: Anam_Lymphom_run2.txt
  → Gespeichert: Anam_Lymphom_run3.txt
Verarbeite: Anam_Cholezystitis.txt
  → Gespeichert: Anam_Cholezystitis_run1.txt
  → Gespeichert: Anam_Cholezys

In [ ]:
import os
import re
import json

input_folder = "output_temp"
output_jsonl_path = "outputs.jsonl"

model_name = "MedGemma"
entry_type = "output"

pattern = re.compile(r"^(?P<audio_id>.+)_run(?P<run_id>\d+)\.txt$", re.IGNORECASE)

experiment_id = 1

with open(output_jsonl_path, "w", encoding="utf-8") as outfile:

    for filename in sorted(os.listdir(input_folder)):
        if not filename.lower().endswith(".txt"):
            continue

        match = pattern.match(filename)
        if not match:
            continue

        audio_id = match.group("audio_id")
        run_id = match.group("run_id")

        file_path = os.path.join(input_folder, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()

        record = {
            "experiment_id": experiment_id,
            "audio_id": audio_id,
            "model_name": model_name,
            "run_id": run_id,
            "type": entry_type,
            "text": text
        }

        # 👉 jede Zeile ein JSON-Objekt
        outfile.write(json.dumps(record, ensure_ascii=False) + "\n")

        experiment_id += 1

print("✅ Fertig! JSONL erstellt.")

✅ Fertig! JSONL erstellt.


In [ ]:
import os
import json

input_folder = "input_temp"
output_jsonl_path = "inputs.jsonl"

model_name = "unknown" 
entry_type = "input"

experiment_id = 1

with open(output_jsonl_path, "w", encoding="utf-8") as outfile:

    for filename in sorted(os.listdir(input_folder)):
        if not filename.lower().endswith(".txt"):
            continue

        file_path = os.path.join(input_folder, filename)

        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()

        # audio_id = Dateiname ohne .txt
        audio_id = os.path.splitext(filename)[0]

        record = {
            "experiment_id": experiment_id,
            "audio_id": audio_id,
            "model_name": model_name,
            "run_id": "0",  
            "type": entry_type,
            "text": text
        }

        outfile.write(json.dumps(record, ensure_ascii=False) + "\n")

        experiment_id += 1

print("✅ inputs.jsonl erstellt!")

✅ inputs.jsonl erstellt!


In [ ]:
import json
import re

input_path = "inputs.jsonl"
output_path = "inputs_clean.jsonl"

def clean_text(text):
    text = re.sub(r"\s*\n+\s*", " ", text)
    text = text.replace("\\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

with open(input_path, "r", encoding="utf-8") as infile, \
     open(output_path, "w", encoding="utf-8") as outfile:

    for line in infile:
        data = json.loads(line)

        text = data.get("text", "")
        data["text"] = clean_text(text)

        outfile.write(json.dumps(data, ensure_ascii=False) + "\n")

print("✅ inputs_clean.jsonl fertig")

✅ inputs_clean.jsonl fertig


In [ ]:
import json
import re

input_path = "outputs.jsonl"
output_path = "outputs_clean.jsonl"

def clean_text(text):
    # 🔥 Intro entfernen
    text = re.sub(
        r'^\s*(okay[, ]+)?hier ist die zusammenfassung.*?zitate[n]?:\s*',
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    # Zeilenumbrüche entfernen
    text = re.sub(r"\s*\n+\s*", " ", text)

    # escaped \n entfernen
    text = text.replace("\\n", " ")

    # Whitespace fixen
    text = re.sub(r"\s+", " ", text).strip()

    return text

with open(input_path, "r", encoding="utf-8") as infile, \
     open(output_path, "w", encoding="utf-8") as outfile:

    for line in infile:
        data = json.loads(line)

        text = data.get("text", "")
        data["text"] = clean_text(text)

        outfile.write(json.dumps(data, ensure_ascii=False) + "\n")

print("✅ outputs_clean.jsonl fertig")

✅ outputs_clean.jsonl fertig


In [ ]:
## Für die Nutzung des Modells Llama

from openai import OpenAI

client = OpenAI(base_url="http://localhost:8002/v1", api_key="none")



In [ ]:
import pandas as pd
from openai import OpenAI
import numpy as np
from tqdm import tqdm  # Fortschrittsanzeige


# =========================
# INPUTS Llama-3.1_8B-Instruct
# =========================


# --- 1️⃣ Daten einlesen ---
df = pd.read_json("inputs_clean.jsonl", lines=True)

# --- 3️⃣ Neue Spalte vorbereiten ---
col_name = "embedding_Llama-3.1_8B-Instruct"
df[col_name] = None

# --- 4️⃣ Embeddings erzeugen ---
embeddings = []

for text in tqdm(df["text"], desc="Berechne Embeddings (INPUTS)"):
    if not isinstance(text, str) or not text.strip():
        embeddings.append(None)
        continue

    response = client.embeddings.create(
        model="meta-llama/Llama-3.1-8B-Instruct",
        input=text
    )
    emb = response.data[0].embedding
    embeddings.append(emb)

df[col_name] = embeddings

# --- 5️⃣ Datei speichern ---
df.to_parquet("emb_inputs.parquet")

print(f"✅ Fertig! {len(df)} INPUTS Texte mit Spalte '{col_name}' gespeichert.")

Berechne Embeddings (INPUTS): 100%|██████████| 17/17 [02:16<00:00,  8.03s/it]

✅ Fertig! 17 INPUTS Texte mit Spalte 'embedding_Llama-3.1_8B-Instruct' gespeichert.


In [ ]:
# =========================
# OUTPUTS Llama-3.1_8B-Instruct
# =========================

# --- 1️⃣ Daten einlesen ---
df = pd.read_json("outputs_clean.jsonl", lines=True)

# --- 3️⃣ Neue Spalte vorbereiten ---
col_name = "embedding_Llama-3.1_8B-Instruct"
df[col_name] = None

# --- 4️⃣ Embeddings erzeugen ---
embeddings = []

for text in tqdm(df["text"], desc="Berechne Embeddings (OUTPUTS)"):
    if not isinstance(text, str) or not text.strip():
        embeddings.append(None)
        continue

    response = client.embeddings.create(
        model="meta-llama/Llama-3.1-8B-Instruct",
        input=text
    )
    emb = response.data[0].embedding
    embeddings.append(emb)

df[col_name] = embeddings

# --- 5️⃣ Datei speichern ---
df.to_parquet("emb_outputs.parquet")

print(f"✅ Fertig! {len(df)} OUTPUTS Texte mit Spalte '{col_name}' gespeichert.")

Berechne Embeddings (OUTPUTS): 100%|██████████| 51/51 [03:40<00:00,  4.32s/it]

✅ Fertig! 51 OUTPUTS Texte mit Spalte 'embedding_Llama-3.1_8B-Instruct' gespeichert.


In [ ]:
df = pd.read_parquet("emb_inputs.parquet")
df

,experiment_id,audio_id,model_name,run_id,type,text,embedding_Llama-3.1_8B-Instruct
0,1,Anam_Aengstliche-Patientin,unknown,0,input,"### Schönen guten Tag, mein Name ist Nina Kule...","[-0.0061825173906981945, -0.007728146854788065..."
1,2,Anam_Appendizitis,unknown,0,input,"Schönen guten Tag, bitte nehmen Sie Platz. Gut...","[0.0037829959765076637, 0.0012421777937561274,..."
2,3,Anam_Cholezystitis,unknown,0,input,"Schönen guten Tag, mein Name ist Nina Kollett....","[0.007391597609966993, -0.000606562418397516, ..."
3,4,Anam_Covid-19_02,unknown,0,input,"Schönen guten Tag, mein Name ist Nina Colette ...","[-0.0071546160615980625, -0.007717971689999103..."
4,5,Anam_Fahrradunfall_02,unknown,0,input,"Schönen guten Tag, mein Name ist Nina Colette ...","[0.005962664727121592, -0.0025150575675070286,..."
5,6,Anam_Fremdanamnese,unknown,0,input,"Schönen guten Tag zusammen, Nina Colette ist m...","[-0.0034050033427774906, -0.009561805985867977..."
6,7,Anam_Fremdanamnese_Saeugling,unknown,0,input,"### Schönen guten Tag, Nina Colette ist mein N...","[0.0016985067632049322, -0.012465073727071285,..."
7,8,Anam_Gastroenteritis,unknown,0,input,"Schönen guten Tag, mein Name ist Nina Kulett. ...","[0.0024882140569388866, -0.0030505224131047726..."
8,9,Anam_Koronarsyndrom,unknown,0,input,"Schönen guten Tag. Guten Tag. Hallo, Nina Cole...","[8.553578663850203e-05, -0.007158687338232994,..."
9,10,Anam_Lymphom,unknown,0,input,"Guten Tag, Kulett ist mein Name. Ich werde jet...","[-0.0035903353709727526, -0.010700606741011143..."


In [ ]:
df = pd.read_parquet("emb_outputs.parquet")
df

,experiment_id,audio_id,model_name,run_id,type,text,embedding_Llama-3.1_8B-Instruct
0,1,Anam_Aengstliche-Patientin,MedGemma,1,output,**Patient:** * **Persönliche Daten:** Hannelor...,"[0.03682839497923851, -0.01676022820174694, 0...."
1,2,Anam_Aengstliche-Patientin,MedGemma,2,output,**Patient:** * **Persönliche Daten:** Hannelor...,"[0.03682839497923851, -0.01676022820174694, 0...."
2,3,Anam_Aengstliche-Patientin,MedGemma,3,output,**Patient:** * **Persönliche Daten:** Hannelor...,"[0.03682839497923851, -0.01676022820174694, 0...."
3,4,Anam_Appendizitis,MedGemma,1,output,**Patient:** * **Persönliche Daten:** Julia Ne...,"[-0.004655799362808466, -0.001314578577876091,..."
4,5,Anam_Appendizitis,MedGemma,2,output,**Patient:** * **Persönliche Daten:** Julia Ne...,"[0.007193032652139664, -0.01092682033777237, 0..."
5,6,Anam_Appendizitis,MedGemma,3,output,**Patient:** * **Persönliche Daten:** Julia Ne...,"[0.010819222778081894, -0.011316023766994476, ..."
6,7,Anam_Cholezystitis,MedGemma,1,output,**Patient:** * **Persönliche Daten:** Laura We...,"[0.01935320347547531, -0.008247103542089462, -..."
7,8,Anam_Cholezystitis,MedGemma,2,output,**Patient:** * **Persönliche Daten:** Laura We...,"[-0.00805704202502966, -0.011312969028949738, ..."
8,9,Anam_Cholezystitis,MedGemma,3,output,**Patient:** * **Persönliche Daten:** Laura We...,"[-0.00805704202502966, -0.011312969028949738, ..."
9,10,Anam_Covid-19_02,MedGemma,1,output,**Patient:** * **Persönliche Daten:** Sebastia...,"[0.0030461850110441446, -0.01311362162232399, ..."


In [ ]:


import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# --- 2️⃣ Modell laden ---
model = SentenceTransformer(
    "jinaai/jina-embeddings-v3",
    trust_remote_code=True,
    device="cuda:2"
)

# =========================
# INPUTS
# =========================

# --- 1️⃣ DataFrame laden ---
df = pd.read_parquet("emb_inputs.parquet")

# --- 3️⃣ Neue Spalte definieren ---
col_name = "embedding_Jina_v3"
df[col_name] = None

# --- 4️⃣ Embeddings berechnen ---
texts = df["text"].fillna("").tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=2,
    normalize_embeddings=True
)

# --- 5️⃣ Embeddings speichern ---
df[col_name] = embeddings.tolist()

# --- 6️⃣ Wieder speichern ---
df.to_parquet("emb_inputs.parquet")

print(f"✅ {len(df)} INPUT Embeddings mit JinaAI v3 gespeichert.")


# =========================
# OUTPUTS
# =========================

# --- 1️⃣ DataFrame laden ---
df = pd.read_parquet("emb_outputs.parquet")

# --- 3️⃣ Neue Spalte definieren ---
col_name = "embedding_Jina_v3"
df[col_name] = None

# --- 4️⃣ Embeddings berechnen ---
texts = df["text"].fillna("").tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=2,
    normalize_embeddings=True
)

# --- 5️⃣ Embeddings speichern ---
df[col_name] = embeddings.tolist()

# --- 6️⃣ Wieder speichern ---
df.to_parquet("emb_outputs.parquet")

print(f"✅ {len(df)} OUTPUT Embeddings mit JinaAI v3 gespeichert.")

/home/wattaah/embedding/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed.

✅ 17 INPUT Embeddings mit JinaAI v3 gespeichert.


Batches: 100%|██████████| 26/26 [00:03<00:00,  7.66it/s]


✅ 51 OUTPUT Embeddings mit JinaAI v3 gespeichert.


In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# --- 2️⃣ Modell laden (CPU) ---
model = SentenceTransformer(
    "aari1995/German_Semantic_V3b",
    trust_remote_code=True,
    device="cpu"
)

# =========================
# INPUTS
# =========================

# --- 1️⃣ DataFrame laden ---
df = pd.read_parquet("emb_inputs.parquet")

# --- 3️⃣ Neue Spalte definieren ---
col_name = "embedding_German_Semantic_V3b"
df[col_name] = None

# --- 4️⃣ Embeddings berechnen ---
texts = df["text"].fillna("").tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=8,  
    normalize_embeddings=True
)

# --- 5️⃣ Embeddings speichern ---
df[col_name] = embeddings.tolist()

# --- 6️⃣ Wieder speichern ---
df.to_parquet("emb_inputs.parquet")

print(f"✅ {len(df)} INPUT Embeddings (CPU) gespeichert.")


# =========================
# OUTPUTS
# =========================

# --- 1️⃣ DataFrame laden ---
df = pd.read_parquet("emb_outputs.parquet")

# --- 3️⃣ Neue Spalte definieren ---
col_name = "embedding_German_Semantic_V3b"
df[col_name] = None

# --- 4️⃣ Embeddings berechnen ---
texts = df["text"].fillna("").tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=8,
    normalize_embeddings=True
)

# --- 5️⃣ Embeddings speichern ---
df[col_name] = embeddings.tolist()

# --- 6️⃣ Wieder speichern ---
df.to_parquet("emb_outputs.parquet")

print(f"✅ {len(df)} OUTPUT Embeddings (CPU) gespeichert.")

Batches: 100%|██████████| 3/3 [03:20<00:00, 66.89s/it] 


✅ 17 INPUT Embeddings (CPU) gespeichert.


Batches: 100%|██████████| 7/7 [09:51<00:00, 84.55s/it] 

✅ 51 OUTPUT Embeddings (CPU) gespeichert.


In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# --- 2️⃣ Modell laden (CPU) ---
model = SentenceTransformer(
    "BAAI/bge-m3",
    trust_remote_code=True,
    device="cpu"
)

# =========================
# INPUTS
# =========================

# --- 1️⃣ DataFrame laden ---
df = pd.read_parquet("emb_inputs.parquet")

# --- 3️⃣ Neue Spalte definieren ---
col_name = "embedding_bge_m3"
df[col_name] = None

# --- 4️⃣ Embeddings berechnen ---
texts = df["text"].fillna("").tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=8,
    normalize_embeddings=True
)

# --- 5️⃣ Embeddings speichern ---
df[col_name] = embeddings.tolist()

# --- 6️⃣ Wieder speichern ---
df.to_parquet("emb_inputs.parquet")

print(f"✅ {len(df)} INPUT Embeddings mit BGE-M3 (CPU) gespeichert.")


# =========================
# OUTPUTS
# =========================

# --- 1️⃣ DataFrame laden ---
df = pd.read_parquet("emb_outputs.parquet")

# --- 3️⃣ Neue Spalte definieren ---
col_name = "embedding_bge_m3"
df[col_name] = None

# --- 4️⃣ Embeddings berechnen ---
texts = df["text"].fillna("").tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=8,
    normalize_embeddings=True
)

# --- 5️⃣ Embeddings speichern ---
df[col_name] = embeddings.tolist()

# --- 6️⃣ Wieder speichern ---
df.to_parquet("emb_outputs.parquet")

print(f"✅ {len(df)} OUTPUT Embeddings mit BGE-M3 (CPU) gespeichert.")

Batches: 100%|██████████| 3/3 [00:36<00:00, 12.17s/it]


✅ 17 INPUT Embeddings mit BGE-M3 (CPU) gespeichert.


Batches: 100%|██████████| 7/7 [01:35<00:00, 13.66s/it]

✅ 51 OUTPUT Embeddings mit BGE-M3 (CPU) gespeichert.


In [6]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# --- 2️⃣ Modell laden (CPU) ---
model = SentenceTransformer(
    "Alibaba-NLP/gte-multilingual-base",
    trust_remote_code=True,
    device="cpu"
)

# =========================
# INPUTS
# =========================

# --- 1️⃣ DataFrame laden ---
df = pd.read_parquet("emb_inputs.parquet")

# --- 3️⃣ Neue Spalte definieren ---
col_name = "embedding_gte_multilingual_base"
df[col_name] = None

# --- 4️⃣ Embeddings berechnen ---
texts = df["text"].fillna("").tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=8,
    normalize_embeddings=True
)

# --- 5️⃣ Embeddings speichern ---
df[col_name] = embeddings.tolist()

# --- 6️⃣ Wieder speichern ---
df.to_parquet("emb_inputs.parquet")

print(f"✅ {len(df)} INPUT Embeddings mit GTE Multilingual Base (CPU) gespeichert.")


# =========================
# OUTPUTS
# =========================

# --- 1️⃣ DataFrame laden ---
df = pd.read_parquet("emb_outputs.parquet")

# --- 3️⃣ Neue Spalte definieren ---
col_name = "embedding_gte_multilingual_base"
df[col_name] = None

# --- 4️⃣ Embeddings berechnen ---
texts = df["text"].fillna("").tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=8,
    normalize_embeddings=True
)

# --- 5️⃣ Embeddings speichern ---
df[col_name] = embeddings.tolist()

# --- 6️⃣ Wieder speichern ---
df.to_parquet("emb_outputs.parquet")

print(f"✅ {len(df)} OUTPUT Embeddings mit GTE Multilingual Base (CPU) gespeichert.")

Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Batches: 100%|██████████| 3/3 [00:17<00:00,  5.87s/it]


✅ 17 INPUT Embeddings mit GTE Multilingual Base (CPU) gespeichert.


Batches: 100%|██████████| 7/7 [00:41<00:00,  6.00s/it]

✅ 51 OUTPUT Embeddings mit GTE Multilingual Base (CPU) gespeichert.


In [7]:
df = pd.read_parquet("emb_inputs.parquet")
df

,experiment_id,audio_id,model_name,run_id,type,text,embedding_Llama-3.1_8B-Instruct,embedding_Jina_v3,embedding_German_Semantic_V3b,embedding_bge_m3,embedding_gte_multilingual_base
0,1,Anam_Aengstliche-Patientin,unknown,0,input,"### Schönen guten Tag, mein Name ist Nina Kule...","[-0.0061825173906981945, -0.007728146854788065...","[0.004852294921875, -0.09326171875, 0.04907226...","[0.06100694462656975, 0.007376667112112045, 0....","[-0.0007007320527918637, 0.04056890308856964, ...","[-0.0817495658993721, 0.07547837495803833, 0.0..."
1,2,Anam_Appendizitis,unknown,0,input,"Schönen guten Tag, bitte nehmen Sie Platz. Gut...","[0.0037829959765076637, 0.0012421777937561274,...","[-0.05517578125, -0.0927734375, 0.033203125, 0...","[0.06820317357778549, 0.02592722699046135, 0.0...","[-0.04383749142289162, 0.04217161238193512, -0...","[-0.09140871465206146, 0.06768683344125748, 4...."
2,3,Anam_Cholezystitis,unknown,0,input,"Schönen guten Tag, mein Name ist Nina Kollett....","[0.007391597609966993, -0.000606562418397516, ...","[0.006103515625, -0.10107421875, 0.0048828125,...","[0.05644379183650017, 0.019454196095466614, 0....","[-0.018477005884051323, 0.008662795647978783, ...","[-0.10140867531299591, 0.09464558959007263, -0..."
3,4,Anam_Covid-19_02,unknown,0,input,"Schönen guten Tag, mein Name ist Nina Colette ...","[-0.0071546160615980625, -0.007717971689999103...","[0.043701171875, -0.078125, 0.00384521484375, ...","[0.05870495364069939, 0.010246138088405132, 0....","[0.03267517313361168, -8.197945135179907e-05, ...","[-0.09303078055381775, 0.07374901324510574, -0..."
4,5,Anam_Fahrradunfall_02,unknown,0,input,"Schönen guten Tag, mein Name ist Nina Colette ...","[0.005962664727121592, -0.0025150575675070286,...","[-0.00848388671875, -0.169921875, -0.013854980...","[0.06014307960867882, 0.01604563370347023, 0.0...","[0.010790705680847168, 0.008178460411727428, -...","[-0.06811187416315079, 0.05828385800123215, 0...."
5,6,Anam_Fremdanamnese,unknown,0,input,"Schönen guten Tag zusammen, Nina Colette ist m...","[-0.0034050033427774906, -0.009561805985867977...","[0.0208740234375, -0.08935546875, 0.0444335937...","[0.056498557329177856, 0.010931120254099369, 0...","[-0.015103154815733433, 0.026062561199069023, ...","[-0.08002178370952606, 0.061940133571624756, 0..."
6,7,Anam_Fremdanamnese_Saeugling,unknown,0,input,"### Schönen guten Tag, Nina Colette ist mein N...","[0.0016985067632049322, -0.012465073727071285,...","[-0.01007080078125, -0.1357421875, 0.060058593...","[0.029995104297995567, 0.0004835352301597595, ...","[0.011272919364273548, 0.017035093158483505, -...","[-0.04087716341018677, 0.08053913712501526, 0...."
7,8,Anam_Gastroenteritis,unknown,0,input,"Schönen guten Tag, mein Name ist Nina Kulett. ...","[0.0024882140569388866, -0.0030505224131047726...","[0.040283203125, -0.07421875, 0.06298828125, 0...","[0.055145006626844406, 0.028580937534570694, 0...","[0.028579283505678177, 0.022073350846767426, -...","[-0.10749651491641998, 0.07372027635574341, 0...."
8,9,Anam_Koronarsyndrom,unknown,0,input,"Schönen guten Tag. Guten Tag. Hallo, Nina Cole...","[8.553578663850203e-05, -0.007158687338232994,...","[-0.014404296875, -0.09619140625, -0.033447265...","[0.06199810653924942, 0.009006169624626637, 0....","[0.02025148645043373, 0.0074670878238976, -0.0...","[-0.07904765009880066, 0.08711352199316025, -0..."
9,10,Anam_Lymphom,unknown,0,input,"Guten Tag, Kulett ist mein Name. Ich werde jet...","[-0.0035903353709727526, -0.010700606741011143...","[0.04150390625, -0.08251953125, 0.0595703125, ...","[0.052613515406847, 0.01378888264298439, 0.031...","[0.0005290775443427265, 0.0272641833871603, -0...","[-0.08040020614862442, 0.09794721752405167, -0..."


In [8]:
df = pd.read_parquet("emb_outputs.parquet")
df

,experiment_id,audio_id,model_name,run_id,type,text,embedding_Llama-3.1_8B-Instruct,embedding_Jina_v3,embedding_German_Semantic_V3b,embedding_bge_m3,embedding_gte_multilingual_base
0,1,Anam_Aengstliche-Patientin,MedGemma,1,output,**Patient:** * **Persönliche Daten:** Hannelor...,"[0.03682839497923851, -0.01676022820174694, 0....","[0.020751953125, -0.0849609375, 0.04345703125,...","[0.04339136928319931, -0.011689098551869392, 0...","[-0.007311564404517412, 0.010435337200760841, ...","[-0.03246108442544937, 0.10843551158905029, 0...."
1,2,Anam_Aengstliche-Patientin,MedGemma,2,output,**Patient:** * **Persönliche Daten:** Hannelor...,"[0.03682839497923851, -0.01676022820174694, 0....","[0.02099609375, -0.0849609375, 0.04345703125, ...","[0.04339136928319931, -0.011689098551869392, 0...","[-0.007311564404517412, 0.010435337200760841, ...","[-0.03246108442544937, 0.10843551158905029, 0...."
2,3,Anam_Aengstliche-Patientin,MedGemma,3,output,**Patient:** * **Persönliche Daten:** Hannelor...,"[0.03682839497923851, -0.01676022820174694, 0....","[0.020751953125, -0.0849609375, 0.04345703125,...","[0.04339136928319931, -0.011689098551869392, 0...","[-0.007311564404517412, 0.010435337200760841, ...","[-0.03246108442544937, 0.10843551158905029, 0...."
3,4,Anam_Appendizitis,MedGemma,1,output,**Patient:** * **Persönliche Daten:** Julia Ne...,"[-0.004655799362808466, -0.001314578577876091,...","[-0.017333984375, -0.06787109375, 0.0612792968...","[0.08574385941028595, 0.0196222010999918, 0.00...","[-0.034746140241622925, 0.014552327804267406, ...","[-0.08017493039369583, 0.08182922005653381, -0..."
4,5,Anam_Appendizitis,MedGemma,2,output,**Patient:** * **Persönliche Daten:** Julia Ne...,"[0.007193032652139664, -0.01092682033777237, 0...","[-0.0634765625, -0.07275390625, 0.060302734375...","[0.04896225407719612, 0.007341993506997824, -0...","[-0.05255601555109024, 0.015221941284835339, -...","[-0.09267380833625793, 0.0838605985045433, -0...."
5,6,Anam_Appendizitis,MedGemma,3,output,**Patient:** * **Persönliche Daten:** Julia Ne...,"[0.010819222778081894, -0.011316023766994476, ...","[-0.044677734375, -0.06396484375, 0.0649414062...","[0.03994660824537277, 0.01691027544438839, 0.0...","[-0.05354717746376991, 0.012697061523795128, -...","[-0.09737160801887512, 0.08862213790416718, -0..."
6,7,Anam_Cholezystitis,MedGemma,1,output,**Patient:** * **Persönliche Daten:** Laura We...,"[0.01935320347547531, -0.008247103542089462, -...","[0.02001953125, -0.032470703125, 0.033203125, ...","[0.040756866335868835, 0.011628600768744946, -...","[-0.016513381153345108, 0.007742623798549175, ...","[-0.10306958109140396, 0.08921503275632858, -0..."
7,8,Anam_Cholezystitis,MedGemma,2,output,**Patient:** * **Persönliche Daten:** Laura We...,"[-0.00805704202502966, -0.011312969028949738, ...","[0.0167236328125, -0.0302734375, 0.02709960937...","[0.04360228404402733, 0.0041997297666966915, -...","[-0.0099851218983531, 0.016561400145292282, -0...","[-0.08911996334791183, 0.09405282139778137, -0..."
8,9,Anam_Cholezystitis,MedGemma,3,output,**Patient:** * **Persönliche Daten:** Laura We...,"[-0.00805704202502966, -0.011312969028949738, ...","[0.01708984375, -0.0302734375, 0.0267333984375...","[0.04360228404402733, 0.0041997297666966915, -...","[-0.0099851218983531, 0.016561400145292282, -0...","[-0.08911996334791183, 0.09405282139778137, -0..."
9,10,Anam_Covid-19_02,MedGemma,1,output,**Patient:** * **Persönliche Daten:** Sebastia...,"[0.0030461850110441446, -0.01311362162232399, ...","[0.0537109375, -0.046630859375, 0.003097534179...","[0.05733107402920723, -0.011462860740721226, 0...","[0.03661473095417023, 0.006658365484327078, -0...","[-0.08020807057619095, 0.07832314819097519, -0..."


## Kosinusähnlichkeit

In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import json

# --- Daten laden ---
df_in = pd.read_parquet("emb_inputs.parquet")
df_out = pd.read_parquet("emb_outputs.parquet")

# --- Embedding-Spalten ---
embedding_cols = [col for col in df_in.columns if col.startswith("embedding_")]

all_results = []

for emb_col in embedding_cols:
    print(f"Berechne: {emb_col}")

    df_in_valid = df_in[df_in[emb_col].notna()]
    df_out_valid = df_out[df_out[emb_col].notna()]

    for audio_id in df_in_valid["audio_id"].unique():
        ref = df_in_valid[df_in_valid["audio_id"] == audio_id]
        others = df_out_valid[df_out_valid["audio_id"] == audio_id]

        if ref.empty or others.empty:
            continue

        ref_emb = np.array(ref.iloc[0][emb_col], dtype=np.float32).reshape(1, -1)

        for _, row in others.iterrows():
            out_emb = np.array(row[emb_col], dtype=np.float32).reshape(1, -1)
            sim = cosine_similarity(ref_emb, out_emb)[0, 0]

            # 👉 einzelne Werte speichern
            all_results.append({
                "embedding_model": emb_col,
                "audio_id": audio_id,
                "run_id": str(row["run_id"]),
                "similarity": float(sim)
            })

res_df = pd.DataFrame(all_results)
# =========================
# 🧮 Mean pro Modell
# =========================
model_mean = (
    res_df.groupby("embedding_model")["similarity"]
    .mean()
    .sort_values(ascending=False)
)

print("\n🏆 Mean pro Embedding-Modell:")
print(model_mean)


# =========================
# ✅ Einzelwerte als JSONL speichern (eine Zeile pro Eintrag)
# =========================
jsonl_path = "cosine_detailed.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for rec in all_results:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"✅ Einzelwerte gespeichert: {jsonl_path}")


Berechne: embedding_Llama-3.1_8B-Instruct
Berechne: embedding_Jina_v3
Berechne: embedding_German_Semantic_V3b
Berechne: embedding_bge_m3
Berechne: embedding_gte_multilingual_base

🏆 Mean pro Embedding-Modell:
embedding_model
embedding_German_Semantic_V3b      0.947855
embedding_gte_multilingual_base    0.848271
embedding_bge_m3                   0.768216
embedding_Jina_v3                  0.730481
embedding_Llama-3.1_8B-Instruct    0.597489
Name: similarity, dtype: float64
✅ Einzelwerte gespeichert: cosine_detailed.jsonl


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import itertools

# --- Daten laden ---
df = pd.read_parquet("emb_outputs.parquet")

# --- alle Embedding-Spalten ---
embedding_cols = [col for col in df.columns if col.startswith("embedding_")]

all_results = []

for emb_col in embedding_cols:
    print(f"Berechne Konsistenz für: {emb_col}")

    df_valid = df[df[emb_col].notna()]

    audio_means = []
    audio_stds_within = []  # <- Streuung der 3 Paar-Sims innerhalb jedes audios

    for audio_id, subset in df_valid.groupby("audio_id"):
        if len(subset) < 2:
            continue

        embeddings = [np.array(e).reshape(1, -1) for e in subset[emb_col]]
        pairs = list(itertools.combinations(embeddings, 2))
        sims = [cosine_similarity(a, b)[0, 0] for a, b in pairs]

        audio_means.append(float(np.mean(sims)))
        audio_stds_within.append(float(np.std(sims)))  

    if len(audio_means) > 0:
        all_results.append({
            "embedding_model": emb_col,
            "mean_consistency": float(np.mean(audio_means)),
            "std_consistency_within_audio": float(np.mean(audio_stds_within)), 
            "n_audios": int(len(audio_means))
        })

consistency_df = pd.DataFrame(all_results).sort_values("mean_consistency", ascending=False)

print("\n🏆 Konsistenz-Ranking (within-audio std):")
print(consistency_df)

Berechne Konsistenz für: embedding_Llama-3.1_8B-Instruct
Berechne Konsistenz für: embedding_Jina_v3
Berechne Konsistenz für: embedding_German_Semantic_V3b
Berechne Konsistenz für: embedding_bge_m3
Berechne Konsistenz für: embedding_gte_multilingual_base

🏆 Konsistenz-Ranking (within-audio std):
                   embedding_model  mean_consistency  \
1                embedding_Jina_v3          0.981940   
4  embedding_gte_multilingual_base          0.980300   
2    embedding_German_Semantic_V3b          0.960249   
3                 embedding_bge_m3          0.945573   
0  embedding_Llama-3.1_8B-Instruct          0.758160   

   std_consistency_within_audio  n_audios  
1                      0.010636        17  
4                      0.011721        17  
2                      0.023117        17  
3                      0.027960        17  
0                      0.113606        17  


In [ ]:
import json
import itertools
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# --- Daten laden ---
df = pd.read_parquet("emb_outputs.parquet")

# --- alle Embedding-Spalten ---
embedding_cols = [col for col in df.columns if col.startswith("embedding_")]

all_results = []
all_pairwise = []  # <- ALLE Einzelwerte (cosine similarities) pro Audio & Paar

for emb_col in embedding_cols:
    print(f"Berechne Konsistenz für: {emb_col}")

    df_valid = df[df[emb_col].notna()]

    audio_means = []
    audio_stds_within = []  # Streuung der Pairwise-Sims innerhalb jedes audios

    for audio_id, subset in df_valid.groupby("audio_id"):
        if len(subset) < 2:
            continue

        embeddings = [np.array(e).reshape(1, -1) for e in subset[emb_col]]
        pairs = list(itertools.combinations(embeddings, 2))

        sims = []
        for i, (a, b) in enumerate(pairs):
            sim = cosine_similarity(a, b)[0, 0]
            sims.append(sim)

            # ✅ Einzelwert speichern
            all_pairwise.append({
                "embedding_model": emb_col,
                "audio_id": str(audio_id),  # als str für JSON-Sicherheit
                "pair_id": int(i),
                "cosine_similarity": float(sim),
                "n_runs_in_audio": int(len(subset)) 
            })

        audio_means.append(float(np.mean(sims)))
        audio_stds_within.append(float(np.std(sims)))

    if len(audio_means) > 0:
        all_results.append({
            "embedding_model": emb_col,
            "mean_consistency": float(np.mean(audio_means)),
            "std_consistency_within_audio": float(np.mean(audio_stds_within)),
            "n_audios": int(len(audio_means))
        })

# --- DataFrames bauen ---
consistency_df = pd.DataFrame(all_results).sort_values("mean_consistency", ascending=False)
pairwise_df = pd.DataFrame(all_pairwise)

print("\n🏆 Konsistenz-Ranking (within-audio std):")
print(consistency_df)

print("\n🔍 Beispiel Einzelwerte:")
print(pairwise_df.head())

# --- JSON speichern ---
# 1) Einzelwerte als JSON (Records)
pairwise_df.to_json("pairwise_results.json", orient="records", indent=2)

# 2) Summary als JSON
consistency_df.to_json("consistency_summary.json", orient="records", indent=2)

# 3)Beides zusammen in einer Datei
output = {
    "summary": consistency_df.to_dict(orient="records"),
    "pairwise": pairwise_df.to_dict(orient="records")
}

with open("results.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("\n✅ Gespeichert: pairwise_results.json, consistency_summary.json, results.json")

Berechne Konsistenz für: embedding_Llama-3.1_8B-Instruct
Berechne Konsistenz für: embedding_Jina_v3
Berechne Konsistenz für: embedding_German_Semantic_V3b
Berechne Konsistenz für: embedding_bge_m3
Berechne Konsistenz für: embedding_gte_multilingual_base

🏆 Konsistenz-Ranking (within-audio std):
                   embedding_model  mean_consistency  \
1                embedding_Jina_v3          0.981940   
4  embedding_gte_multilingual_base          0.980300   
2    embedding_German_Semantic_V3b          0.960249   
3                 embedding_bge_m3          0.945573   
0  embedding_Llama-3.1_8B-Instruct          0.758160   

   std_consistency_within_audio  n_audios  
1                      0.010636        17  
4                      0.011721        17  
2                      0.023117        17  
3                      0.027960        17  
0                      0.113606        17  

🔍 Beispiel Einzelwerte:
                   embedding_model                    audio_id  pair_id  \
0  